# DashAnalysis Tool - Updated with New Slice Data Methods

This notebook demonstrates the enhanced DashAnalysis tool's data loading and slicing capabilities using the new slice data methods.

The basic workflow is:
```python
data = da.load_data(path)
data.points
data.intensities
sl_data = da.slice_data(data)
lc_data = da.line_cut(data)
```

**New Enhanced Features:**
- **Volume slicing**: Uses PyVista with a plane defined by `normal` and `origin`
- **Point cloud slicing**: Selects points near a specified plane and interpolates onto that plane
- **HKL axis specification**: Custom HKL coordinate systems with `axes` parameter
- **Slab slicing**: Creates thick slices with specified thickness
- **Interactive line cuts**: Draggable endpoints for real-time analysis
- **Enhanced visualization**: Better colormap control and opacity ranges

In [ ]:
# Import and setup
# Import dash_analysis module with proper error handling
dash = None
try:
    import dash_analysis as dash
except ModuleNotFoundError:
    try:
        import sys
        import os
        # Add parent directory to path for notebook imports
        notebook_dir = os.getcwd()
        parent_dir = os.path.dirname(os.path.dirname(notebook_dir))
        if parent_dir not in sys.path:
            sys.path.append(parent_dir)
        from utils import dash_analysis as dash
        print("Successfully imported dash_analysis from utils")
    except ImportError as e:
        print(f"Failed to import dash_analysis: {e}")


import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt

pv.set_jupyter_backend('html')
da = dash.DashAnalysis()

## Data Loading and Metadata

Load data from HDF5 file or generate synthetic data for demonstration.

In [ ]:
# Try loading a real HDF5 file; fall back to synthetic data if unavailable
filename = "/home/beams/OODIASEIGIEHON/DashPVA/selective_compression.h5"  # change to your file path as needed
# filename = "/home/beams/OODIASEIGIEHON/DashPVA/dummy/DUMMY_HKL_3D.h5"  # change to your file path as needed

data = da.load_data(filename)

# Show metadata
da.show_meta(filename)

## Basic 2D Slice Examples

Create 2D slices using the new slice_data method with different HKL orientations.

In [ ]:
# Basic HK slice (traditional approach)
sl_data_hk = da.slice_data(
    data=data,
    hkl='HL',  # HK plane preset
    shape=(256, 256),
    show=True
)

print(f"HK slice created with {sl_data_hk.n_points} points")

In [ ]:
# HL slice with specific origin
sl_data_hl = da.slice_data(
    data=data,
    hkl=(2.0, 0.5, 0.8),  # Specific HKL origin point
    normal=(0, 1, 0),     # HL plane normal
    shape=(256, 256),
    show=True
)

print(f"HL slice created with {sl_data_hl.n_points} points")

## Advanced HKL Axis Specification

Use the new `axes` parameter to define custom HKL coordinate systems.

In [ ]:
# Custom HKL axes: X=H/2, Y=K
sl_custom1 = da.slice_data(
    data=data,
    axes=((3, 1, 0), (1, 1, 0)),  # u_hkl=(H/2), v_hkl=(K)
    shape=(400, 400),
    show=True
)

print("Custom slice with X=H/2, Y=K axes")

In [ ]:
# Mixed HKL axes: X=H+K, Y=L
sl_custom2 = da.slice_data(
    data=data,
    axes=((1, 1, 0), (0, 0, 1)),  # u_hkl=(H+K), v_hkl=(L)
    shape=(400, 400),
    show=True
)

print("Custom slice with X=H+K, Y=L axes")

## 3D Slab Slice with Thickness

Create thick slices that include points within a specified distance from the plane.

In [ ]:
# 3D slab slice with thickness - this will take about 8-12 seconds
sl_3d_data = da.slice_data(
    data=(data.points, data.intensities), 
    hkl='HL',  # HL plane
    shape=(800, 800), 
    slab_thickness=2.0,  # Include points within ±2.0 units of the plane
    spacing=(0.5, 0.5, 0.5),
    show=False  # We'll visualize this in 3D below
)

print(f"3D slab slice shape: {sl_3d_data.points.shape}")
print(f"Intensity range: {sl_3d_data['intensity'].min():.2f} - {sl_3d_data['intensity'].max():.2f}")

## 3D Visualization with PyVista

Visualize the 3D slice data using PyVista's interactive rendering.

In [ ]:
# Create 3D visualization with enhanced rendering
p = pv.Plotter(notebook=True)
p.add_axes(xlabel='H', ylabel='K', zlabel='L')

# Add point cloud with transparency
p.add_mesh(
    sl_3d_data, 
    scalars='intensity',
    render_points_as_spheres=True, 
    point_size=2.0,
    opacity=0.15, 
    cmap='viridis', 
    name='points'
)

# Add the main slice surface with different colormap
p.add_mesh(
    sl_3d_data, 
    scalars='intensity', 
    cmap='magma', 
    show_scalar_bar=True, 
    name='slice'
)

p.show()

## Enhanced Slice Visualization

Use the improved show_slice method with better control over display parameters.

In [ ]:
# Show slice with custom intensity limits and colormap
da.show_slice(
    sl_3d_data, 
    cmap='jet', 
    clim=(0, 10000),
    min_intensity=100,  # Filter out low intensities
    max_intensity=50000  # Cap high intensities
)

## Line Cut Examples - Traditional Methods

Demonstrate different types of line cuts using the enhanced line_cut method.

In [ ]:
# Zero line cut - horizontal line at fixed V coordinate
lc_data_zero = da.line_cut(
    "zero", 
    param=(2.85, "x"),  # V=2.85, traverse U direction
    vol=sl_3d_data,
    n_samples=512,
    show=True
)

print(f"Zero line cut - Distance range: {lc_data_zero['distance'].min():.2f} to {lc_data_zero['distance'].max():.2f}")
print(f"Intensity range: {lc_data_zero['intensity'].min():.2f} to {lc_data_zero['intensity'].max():.2f}")

In [ ]:
# Infinite line cut - vertical line at fixed U coordinate
lc_data_infinite = da.line_cut(
    "infinite", 
    param=(1.0, "y"),  # U=1.0, traverse V direction
    vol=sl_3d_data,
    n_samples=512,
    width_px=3,  # Average over 3 pixels width
    show=True
)

print(f"Infinite line cut completed with {len(lc_data_infinite['distance'])} samples")

In [ ]:
# Positive diagonal line cut
lc_data_positive = da.line_cut(
    "positive", 
    vol=sl_3d_data, 
    n_samples=1024,  # High resolution sampling
    width_px=5,      # Wider averaging for better statistics
    show=True
)

print(f"Positive diagonal line cut with {len(lc_data_positive['distance'])} high-resolution samples")

In [ ]:
# Custom line cut between two specific points
lc_data_custom = da.line_cut(
    ((1.1, 1.75), (0.93, 3.00)),  # Custom endpoints in physical coordinates
    vol=sl_3d_data, 
    n_samples=256,
    show=True
)

print(f"Custom line cut between points (1.1,1.75) and (0.93,3.00)")
print(f"Endpoints in result: {lc_data_custom['endpoints']}")

## Interactive Line Cuts

Use the new interactive mode for real-time line cut analysis with draggable endpoints.

In [ ]:
# Enable interactive matplotlib backend for draggable line cuts
# Note: This requires an interactive backend like 'widget' or 'notebook'
# Run: %matplotlib widget (if ipympl is installed) or %matplotlib notebook

# Interactive line cut with draggable endpoints
lc_interactive = da.line_cut(
    "positive",  # Start with diagonal
    vol=sl_3d_data,
    interactive=True,  # Enable interactive mode
    n_samples=512,
    width_px=3
)

print("Interactive line cut created - drag the cyan endpoints to update the profile in real-time!")

## Enhanced Point Cloud Visualization

Demonstrate the improved point cloud visualization with filtering and opacity control.

In [ ]:
# Basic point cloud visualization with intensity-based opacity
da.show_point_cloud(
    data.points, 
    data.intensities, 
    clim=(0.0, 1000000.0), 
    opacity=0.1,
    point_size=1.5,
    cmap='viridis'
)

print(f"Maximum intensity: {max(data.intensities):.0f}")
print(f"Minimum intensity: {min(data.intensities):.0f}")

In [ ]:
# Enhanced point cloud with range filtering and variable opacity
da.show_point_cloud(
    data.points, 
    data.intensities, 
    clim=(200, 351998),           # Intensity display range
    hide_out_of_range=True,       # Hide points outside clim range
    cmap='jet', 
    opacity_range=(0.1, 0.8),     # Variable opacity based on intensity
    point_size=2.0,
    render_points_as_spheres=True
)

print("Point cloud with intensity filtering and variable opacity rendering")

## Volume Creation and Visualization

Create a 3D volume from point cloud data and visualize it.

In [ ]:
# Create a 3D volume from the point cloud
volume = da.create_vol(data.points, data.intensities)

print(f"Created volume with dimensions: {volume.dimensions}")
print(f"Volume spacing: {volume.spacing}")
print(f"Volume origin: {volume.origin}")

# Visualize the volume
da.show_vol(volume, cmap='plasma')

## Advanced Slice Data Methods

Demonstrate additional advanced features of the slice_data method.

In [ ]:
# Plane-based slicing with custom normal vector and origin
plane_slice = da.slice_data(
    data=data,
    normal=(1, 1, 0),     # Custom normal vector
    hkl=(1.0, 0.5, 0.2),  # Origin point
    shape=(512, 512),
    spacing=(0.1, 0.1),
    clamp_to_bounds=True,
    show=True
)

print(f"Custom plane slice with normal (1,1,0) created")
print(f"Slice contains {plane_slice.n_points} interpolated points")

In [ ]:
# Multi-directional slicing with full 3-vector specification
multi_slice = da.slice_data(
    data=data,
    axes=((1.0, 1.0, 0.0), (0.0, 0.0, 1.0), (1.0, -1.0, 0.0)),  # u, v, normal vectors
    shape=(400, 400),
    slab_thickness=1.5,
    show=True
)

print("Multi-directional slice with full 3-vector specification")
print(f"Field data keys: {list(multi_slice.field_data.keys())}")

## Working with Slice Return Images

Demonstrate how to work with slice images for further analysis.

In [ ]:
# Get slice as image array for further processing
img, extent = da.show_slice(
    sl_3d_data,
    cmap='coolwarm',
    return_image=True  # Return the image array and extent
)

print(f"Slice image shape: {img.shape}")
print(f"Extent: {extent}")
print(f"Image statistics: min={img.min():.2f}, max={img.max():.2f}, mean={img.mean():.2f}")

# Use the returned image for line cuts
lc_from_image = da.line_cut(
    "zero",
    param=(2.0, "x"),
    vol=(img, extent),  # Pass the image and extent directly
    show=True
)

print("Line cut created from returned slice image")

## Summary

This notebook demonstrated the comprehensive new slice data methods in DashAnalysis:

### Core Features:
1. **Enhanced slice_data method** with multiple input formats and HKL axis control
2. **Custom HKL coordinate systems** using the `axes` parameter
3. **Slab slicing** with thickness control for 3D analysis
4. **Interactive line cuts** with draggable endpoints
5. **Advanced point cloud visualization** with opacity and filtering controls

### New Capabilities:
- **Flexible axis specification**: Define custom HKL coordinate systems
- **Improved interpolation**: Better handling of sparse point clouds
- **Enhanced visualization**: Multiple colormaps, opacity ranges, and filtering
- **Interactive analysis**: Real-time line cut updates with draggable interfaces
- **Volume creation**: Convert point clouds to structured volumes
- **Image-based workflows**: Work with slice images for custom analysis

### Performance Improvements:
- **Adaptive resolution**: Automatic grid sizing based on data density
- **Efficient interpolation**: Optimized radius selection for point cloud interpolation
- **Caching**: Intelligent caching of slice images for line cut operations

The new methods provide significantly more flexibility and control over crystallographic data analysis, making it easier to explore complex datasets and extract meaningful insights.